<a href="https://colab.research.google.com/github/Chanlukita/RAG/blob/main/RAGhehe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

RAG System

In [ ]:
!pip install langchain-groq langchain llama-parse qdrant-client "unstructured[md]" fastembed flashrank langchain-community

In [ ]:
import os
import textwrap
from pathlib import Path

from google.colab import userdata
from IPython.display import Markdown
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import FlashrankRerank
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Qdrant
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from llama_parse import LlamaParse
from dotenv import load_dotenv

load_dotenv()

import nltk
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

def print_response(response):
    response_txt = response["result"]
    for chunk in response_txt.split("\n"):
        if not chunk:
            print()
            continue
        print("\n".join(textwrap.wrap(chunk, 100, break_long_words=False)))

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


Document Parsing

In [ ]:
# !mkdir data
!gdown 1VEwxk_SWVycWO4IKwAm6KumHUjXhvH84 -O "nvidia-report.pdf"

Downloading...
From: https://drive.google.com/uc?id=1VEwxk_SWVycWO4IKwAm6KumHUjXhvH84
To: /content/nvidia-report.pdf
100% 75.5k/75.5k [00:00<00:00, 62.6MB/s]


In [ ]:
instruction = """The provided document is NVIDIA Results for First Quarter Fiscal 2025.
This form provides detailed financial information about the company's
performance for a specific quarter.
It includes unaudited financial statements, management discussion and
analysis, and other relevant disclosures required by the SEC.
It contains many tables."""

parser = LlamaParse(
    api_key=userdata.get('LLAMA_PARSE'),
    result_type="markdown",
    parsing_instruction=instruction,
    max_timeout=5000,
)

llama_parse_documents = await parser.aload_data("/content/nvidia-report.pdf")
parsed_doc = llama_parse_documents[0]

Started parsing the file under job_id 1b9261a6-d694-4dae-9a83-5a6a9ea6dfc9


In [ ]:
!mkdir -p data
document_path = Path("data/parsed_document.md")
with document_path.open("w") as f:
    f.write(parsed_doc.text)

Document Chunking

In [ ]:
loader = UnstructuredMarkdownLoader(document_path)

In [ ]:
loaded_documents = loader.load()

In [ ]:
loaded_documents[:1]

[Document(metadata={'source': 'data/parsed_document.md'}, page_content='NVIDIA Announces Financial Results for First Quarter Fiscal 2025\n\nRecord Quarterly Revenue: $26.0 billion, up 18% from Q4 and up 262% from a year ago.\n\nRecord Data Center Revenue: $22.6 billion, up 23% from Q4 and up 427% from a year ago.\n\nStock Split: Ten-for-one forward stock split effective June 7, 2024.\n\nIncreased Cash Dividend: Quarterly cash dividend raised 150% to $0.01 per share on a post-split basis.\n\nFinancial Highlights for Q1 Fiscal 2025 (in millions, except earnings per share):\n\nMetric Q1 FY25 Q4 FY24 Q1 FY24 Q/Q Change Y/Y Change Revenue $26,044 $22,103 $7,192 Up 18% Up 262% Gross Margin 78.4% 76.0% 64.6% Up 2.4 pts Operating Expenses $3,497 $3,176 $2,508 Up 10% Up 39% Operating Income $16,909 $13,615 $2,140 Up 24% Up 690% Net Income $14,881 $12,285 $2,043 Up 21% Up 628% Diluted Earnings per Share $5.98 $4.93 $0.82 Up 21% Up 629%\n\nManagement Commentary: Jensen Huang, founder and CEO of N

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)

In [ ]:
docs = text_splitter.split_documents(loaded_documents)

In [ ]:
len(docs)

4

In [ ]:
print(docs[0].page_content)

NVIDIA Announces Financial Results for First Quarter Fiscal 2025

Record Quarterly Revenue: $26.0 billion, up 18% from Q4 and up 262% from a year ago.

Record Data Center Revenue: $22.6 billion, up 23% from Q4 and up 427% from a year ago.

Stock Split: Ten-for-one forward stock split effective June 7, 2024.

Increased Cash Dividend: Quarterly cash dividend raised 150% to $0.01 per share on a post-split basis.

Financial Highlights for Q1 Fiscal 2025 (in millions, except earnings per share):


Embedding and Vector Database

In [ ]:
embeddings = FastEmbedEmbeddings(model_name="BAAI/bge-base-en-v1.5")

In [ ]:
qdrant = Qdrant.from_documents(
    docs,
    embeddings,
    # location=":memory:", # Use in-memory storage
    path="./db", # Comment out the path
    collection_name="document_embeddings",
)

In [ ]:
query = "What was NVIDIA's total revenue for Q1 FY2025?"
similar_docs = qdrant.similarity_search_with_score(query)

for doc, score in similar_docs:
    print(f"text: {doc.page_content[:256]}\n")
    print(f"score: {score}")
    print("-" * 80)
    print()

text: NVIDIA Announces Financial Results for First Quarter Fiscal 2025

Record Quarterly Revenue: $26.0 billion, up 18% from Q4 and up 262% from a year ago.

Record Data Center Revenue: $22.6 billion, up 23% from Q4 and up 427% from a year ago.

Stock Split: Ten

score: 0.8446737794662832
--------------------------------------------------------------------------------

text: NVIDIA Announces Financial Results for First Quarter Fiscal 2025

Record Quarterly Revenue: $26.0 billion, up 18% from Q4 and up 262% from a year ago.

Record Data Center Revenue: $22.6 billion, up 23% from Q4 and up 427% from a year ago.

Stock Split: Ten

score: 0.8132709881786262
--------------------------------------------------------------------------------

text: NVIDIA Announces Financial Results for First Quarter Fiscal 2025

Record Quarterly Revenue: $26.0 billion, up 18% from Q4 and up 262% from a year ago.

Record Data Center Revenue: $22.6 billion, up 23% from Q4 and up 427% from a year ago.

Stock Spli

Re-ranking

In [ ]:
compressor = FlashrankRerank(model="ms-marco-MiniLM-L-12-v2")
retriever = qdrant.as_retriever(search_kwargs={"k": 10})
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

In [ ]:
reranked_docs = compression_retriever.invoke(query)
len(reranked_docs)

3

In [ ]:
for doc in reranked_docs:
    print(f"id: {doc.metadata['_id']}\n")
    print(f"text: {doc.page_content[:256]}\n")
    print(f"score: {doc.metadata['relevance_score']}")
    print("-" * 80)
    print()

id: 9253e4996c6a41e18762b59272eddcd3

text: NVIDIA Announces Financial Results for First Quarter Fiscal 2025

Record Quarterly Revenue: $26.0 billion, up 18% from Q4 and up 262% from a year ago.

Record Data Center Revenue: $22.6 billion, up 23% from Q4 and up 427% from a year ago.

Stock Split: Ten

score: 0.9950607419013977
--------------------------------------------------------------------------------

id: ed547a542f8841779e424d5a6dbc3afd

text: NVIDIA Announces Financial Results for First Quarter Fiscal 2025

Record Quarterly Revenue: $26.0 billion, up 18% from Q4 and up 262% from a year ago.

Record Data Center Revenue: $22.6 billion, up 23% from Q4 and up 427% from a year ago.

Stock Split: Ten

score: 0.9950607419013977
--------------------------------------------------------------------------------

id: 9a44e0660d524663a8dd8edd99be2173

text: NVIDIA Announces Financial Results for First Quarter Fiscal 2025

Record Quarterly Revenue: $26.0 billion, up 18% from Q4 and up 262% fr

Q&A Over Document

In [ ]:
llm = ChatGroq(temperature=0, model_name="llama-3.3-70b-versatile", api_key=userdata.get('GROQ_API_KEY'))

In [ ]:
prompt_template = """
Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: {context}
Question: {question}

Answer the question and provide additional helpful information,
based on the pieces of information, if applicable. Be succinct.

Responses should be properly formatted to be easily read.
"""

prompt = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

In [ ]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=compression_retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt, "verbose": True},
)

In [ ]:
response = qa.invoke("What is NVIDIA's gross margin for Q1 FY2025??")



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
 
Context: NVIDIA Announces Financial Results for First Quarter Fiscal 2025

Record Quarterly Revenue: $26.0 billion, up 18% from Q4 and up 262% from a year ago.

Record Data Center Revenue: $22.6 billion, up 23% from Q4 and up 427% from a year ago.

Stock Split: Ten-for-one forward stock split effective June 7, 2024.

Increased Cash Dividend: Quarterly cash dividend raised 150% to $0.01 per share on a post-split basis.

Financial Highlights for Q1 Fiscal 2025 (in millions, except earnings per share):

Metric Q1 FY25 Q4 FY24 Q1 FY24 Q/Q Change Y/Y Change Revenue $26,044 $22,103 $7,192 Up 18% Up 262% Gross Margin 78.4% 76.0% 64.6% Up 2.4 pts Operating Expenses $3,497 $3,176 $2,508 Up 10% Up 39% Operating Income $16,909 $1

In [ ]:
response['result']

"**NVIDIA's Gross Margin for Q1 FY2025:** \n78.4%\n\n**Additional Information:**\nThis represents an increase of 2.4 percentage points from Q4 FY2024 (76.0%) and 13.8 percentage points from Q1 FY2024 (64.6%)."

In [ ]:
response = qa.invoke("What new partnerships did NVIDIA announce in the Data Center segment?")



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
 
Context: Management Commentary: Jensen Huang, founder and CEO of NVIDIA, stated that the next industrial revolution has begun, with companies and countries partnering with NVIDIA to transition traditional data centers to accelerated computing and AI factories. The demand for generative AI training and inference is driving significant growth across various sectors, including cloud service providers, consumer internet companies, and healthcare.

NVIDIA Announces Financial Results for First Quarter Fiscal 2025

Record Quarterly Revenue: $26.0 billion, up 18% from Q4 and up 262% from a year ago.

Record Data Center Revenue: $22.6 billion, up 23% from Q4 and up 427% from a year ago.

Stock Split: Ten-for-one forward stock s

In [ ]:
response["source_documents"]

[Document(metadata={'id': 0, 'relevance_score': np.float32(0.98940873), 'source': 'data/parsed_document.md', '_id': '280decc57add413f8d00b4bdcafa2e30', '_collection_name': 'document_embeddings'}, page_content='Management Commentary: Jensen Huang, founder and CEO of NVIDIA, stated that the next industrial revolution has begun, with companies and countries partnering with NVIDIA to transition traditional data centers to accelerated computing and AI factories. The demand for generative AI training and inference is driving significant growth across various sectors, including cloud service providers, consumer internet companies, and healthcare.'),
 Document(metadata={'id': 1, 'relevance_score': np.float32(0.92393386), 'source': 'data/parsed_document.md', '_id': 'ed547a542f8841779e424d5a6dbc3afd', '_collection_name': 'document_embeddings'}, page_content='NVIDIA Announces Financial Results for First Quarter Fiscal 2025\n\nRecord Quarterly Revenue: $26.0 billion, up 18% from Q4 and up 262% fro

In [ ]:
print_response(response)

**NVIDIA's New Partnerships in the Data Center Segment:**
The provided information does not specifically mention any new partnerships announced by NVIDIA in
the Data Center segment. However, it does mention that companies and countries are partnering with
NVIDIA to transition traditional data centers to accelerated computing and AI factories, driven by
the demand for generative AI training and inference.

**Additional Information:**
NVIDIA's Data Center revenue has seen significant growth, with a 23% increase from Q4 and a 427%
increase from a year ago, reaching a record $22.6 billion. This growth is attributed to the demand
for generative AI training and inference across various sectors, including cloud service providers,
consumer internet companies, and healthcare.


Perbandingan Dua Sistem Retrieval

In [ ]:
import re

def normalize_answer(s):
    """Lowercase, remove punctuation, and extra whitespace."""
    return re.sub(r'\s+', ' ', re.sub(r'[^\w\s]', '', s.lower())).strip()

def compute_f1(pred, true):
    pred_tokens = normalize_answer(pred).split()
    true_tokens = normalize_answer(true).split()
    common = set(pred_tokens) & set(true_tokens)
    if not common:
        return 0
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(true_tokens)
    f1 = 2 * (precision * recall) / (precision + recall)
    return f1

In [ ]:
from langchain.chains import RetrievalQA

# === Inisialisasi dua sistem QA ===
qa_normal = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

qa_reranked = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=ContextualCompressionRetriever(
        base_compressor=FlashrankRerank(model="ms-marco-MiniLM-L-12-v2"),
        base_retriever=retriever
    ),
    return_source_documents=True
)

qa_pairs = [
    {
        "question": "What is NVIDIA's gross margin for Q1 FY2025?",
        "answer": "Gross margin for the first quarter of fiscal 2025 was 78.4%"
    },
    {
        "question": "What new partnerships did NVIDIA announce in the Data Center segment?",
        "answer": "NVIDIA announced new partnerships with Dell, HPE, and Microsoft"
    },
    {
        "question": "How much revenue did NVIDIA's gaming segment generate?",
        "answer": "NVIDIA’s Gaming revenue was $2.6 billion"
    },
    {
        "question": "What is the percentage growth of the Data Center segment YoY?",
        "answer": "Data Center revenue grew 427% year-over-year"
    }
]

In [ ]:
def evaluate(qa_chain, qa_pairs):
    results = []
    for pair in qa_pairs:
        pred = qa_chain.invoke(pair["question"])["result"]
        f1 = compute_f1(pred, pair["answer"])
        rouge = compute_rouge(pred, pair["answer"])
        results.append({
            "question": pair["question"],
            "prediction": pred,
            "f1_score": f1,
            "rougeL": rouge
        })
    return results


normal_results = evaluate(qa_normal, qa_pairs)
reranked_results = evaluate(qa_reranked, qa_pairs)

for i, pair in enumerate(qa_pairs):
    print(f"\nQ: {pair['question']}")
    print(f"- Ground Truth: {pair['answer']}")
    print(f"- Normal RAG F1: {normal_results[i]['f1_score']:.2f}")
    print(f"- Reranked RAG F1: {reranked_results[i]['f1_score']:.2f}")
    print(f"- Normal Prediction: {normal_results[i]['prediction']}")
    print(f"- Reranked Prediction: {reranked_results[i]['prediction']}")



Q: What is NVIDIA's gross margin for Q1 FY2025?
- Ground Truth: Gross margin for the first quarter of fiscal 2025 was 78.4%
- Normal RAG F1: 0.42
- Reranked RAG F1: 0.42
- Normal Prediction: NVIDIA's gross margin for Q1 FY2025 is 78.4%.
- Reranked Prediction: NVIDIA's gross margin for Q1 FY2025 is 78.4%.

Q: What new partnerships did NVIDIA announce in the Data Center segment?
- Ground Truth: NVIDIA announced new partnerships with Dell, HPE, and Microsoft
- Normal RAG F1: 0.21
- Reranked RAG F1: 0.21
- Normal Prediction: The text does not mention any new partnerships announced by NVIDIA in the Data Center segment. It only mentions that companies and countries are partnering with NVIDIA to transition traditional data centers to accelerated computing and AI factories, but it does not provide specific details about new partnerships.
- Reranked Prediction: The text does not mention any new partnerships announced by NVIDIA in the Data Center segment. It only mentions that companies and cou

In [ ]:
!pip install rouge-score

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def compute_rouge(pred, ref):
    scores = scorer.score(ref, pred)
    return scores['rougeL'].fmeasure


In [ ]:
import pandas as pd

f1_rouge_table = pd.DataFrame({
    "Question": [pair["question"] for pair in qa_pairs],
    "F1 - Normal RAG": [r["f1_score"] for r in normal_results],
    "ROUGE-L - Normal RAG": [r["rougeL"] for r in normal_results],
    "F1 - Reranked RAG": [r["f1_score"] for r in reranked_results],
    "ROUGE-L - Reranked RAG": [r["rougeL"] for r in reranked_results],
})

display(f1_rouge_table)


,Question,F1 - Normal RAG,ROUGE-L - Normal RAG,F1 - Reranked RAG,ROUGE-L - Reranked RAG
0,What is NVIDIA's gross margin for Q1 FY2025?,0.421053,0.454545,0.421053,0.454545
1,What new partnerships did NVIDIA announce in t...,0.210526,0.140351,0.210526,0.140351
2,How much revenue did NVIDIA's gaming segment g...,0.136364,0.166667,0.171429,0.153846
3,What is the percentage growth of the Data Cent...,0.750000,0.800000,0.444444,0.454545
